<a href="https://colab.research.google.com/github/vrinda-162/Accenture-1L-contract-review-challenge/blob/main/notebooks/Google_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import json
import pandas as pd
import re
from sklearn.model_selection import train_test_split

In [ ]:
## Task 1: Clean & Split Data — Katie & Aarnav

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!unzip -o "/content/cuad-v1.zip" -d "/content/cuad_data"

Archive:  /content/cuad-v1.zip
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of /content/cuad-v1.zip or
        /content/cuad-v1.zip.zip, and cannot find /content/cuad-v1.zip.ZIP, period.


In [ ]:
!ls -R /content/cuad_data

ls: cannot access '/content/cuad_data': No such file or directory


In [ ]:
!unzip -o "/content/cuad-v1.zip" -d "/content/cuad_data"
# Load the full CUAD dataset (all 510 contracts)
with open("/content/cuad_data/CUADv1.json", "r") as f:
    data = json.load(f)
print(f"Total contracts: {len(data['data'])}")
# we're building two separate tables from this one file:
# 1. contracts -- the contract name + its full text (for chunking later)
# 2. annotations -- every labeled clause span (for classification labels)
contracts = []
annotations = []

for contract in data["data"]:
    title = contract["title"]

    # each contract has one paragraph with a "context" field
    # that holds the entire contract text (can be 1,000 to 300,000+ chars)
    context = contract["paragraphs"][0]["context"]

    # one row per contract: just the name and full text
    contracts.append({"contract_title": title, "context": context})

    # now loop through the question-answer pairs for this contract
    # each qa asks "does category X appear in this contract?"
    # if yes, the answer contains the exact text the expert highlighted
    for qa in contract["paragraphs"][0]["qas"]:

        # the qa id looks like "ContractName__Parties_0"
        # split on "__" and take the last part to get the category name
        raw_category = qa["id"].split("__")[-1] if "__" in qa["id"] else qa.get("id", "Unknown")
        question = qa["question"]
        is_impossible = qa.get("is_impossible", False)

        # skip if no answers -- means this category doesn't exist in this contract
        if qa.get("answers"):
            for ans in qa["answers"]:
                annotations.append({
                    "contract_title": title,
                    "category_raw": raw_category,
                    "clause_text": ans["text"],          # the exact clause the expert highlighted
                    "answer_start": ans["answer_start"],  # where in the contract text it starts (char position)
                    "question": question,
                    "is_impossible": is_impossible
                })

# turn both lists into pandas tables
contracts_df = pd.DataFrame(contracts)
annotations_df = pd.DataFrame(annotations)

Archive:  /content/cuad-v1.zip
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of /content/cuad-v1.zip or
        /content/cuad-v1.zip.zip, and cannot find /content/cuad-v1.zip.ZIP, period.


FileNotFoundError: [Errno 2] No such file or directory: '/content/cuad_data/CUADv1.json'

In [ ]:
# are any cells empty?
print("Missing values:")
print(annotations_df.isnull().sum())

# are any rows exact copies of another row?
print(f"\nDuplicate rows: {annotations_df.duplicated().sum()}")

# did any expert accidentally highlight nothing (blank text)?
blank = annotations_df[annotations_df["clause_text"].str.strip() == ""]
print(f"Blank clause text: {len(blank)}")

# did any expert accidentally highlight just 1-2 characters?
# these are bad labels -- things like "." tagged as a date
# or "GA" tagged as a party name
short = annotations_df[annotations_df["clause_text"].str.strip().str.len() <= 2]
print(f"Short answers (<=2 chars): {len(short)}")
if len(short) > 0:
    print(short[["contract_title", "category_raw", "clause_text"]])

# now actually fix the problems we found

# remember how many rows we started with
before = len(annotations_df)

# drop the rows where the clause text is 2 characters or less
annotations_df = annotations_df[annotations_df["clause_text"].str.strip().str.len() > 2]

# drop exact duplicate rows just in case
annotations_df = annotations_df.drop_duplicates().reset_index(drop=True)

# clean up extra spaces at the start/end of clause text
annotations_df["clause_text"] = annotations_df["clause_text"].str.strip()

# show how many rows we removed
print(f"\nAnnotations: {before} -> {len(annotations_df)} ({before - len(annotations_df)} removed)")






In [ ]:
# right now the categories have numbers at the end like Parties_0, Parties_1, Parties_2
# those numbers just mean "first time Parties appeared in this contract", "second time", etc.
# they're all the same category -- just different occurrences
# we need to remove those numbers so they all become just "Parties"
# without this, the model would think there are 408 categories instead of the correct 41

def get_base_category(cat):
    # looks for a pattern like _0, _1, _2, _10 at the very end of the string
    # and removes it, leaving just the category name
    return re.sub(r'_\d+$', '', cat)

# run that function on every row's category and save it as a new column called "category"
# we keep "category_raw" around in case anyone needs the original numbering later
annotations_df["category"] = annotations_df["category_raw"].apply(get_base_category)

# this should print 41 -- the CUAD dataset has exactly 41 clause categories
# if it prints anything else, something went wrong with the stripping
print(f"Unique categories: {annotations_df['category'].nunique()}")

# show how many examples each category has
# notice the imbalance: Parties has ~2000 examples while some categories have under 30
# Task 2 (Vrinda and Hadent) will visualize this in their EDA
print(annotations_df["category"].value_counts())

In [ ]:
# the full contract text in contracts_df has formatting issues from the original PDF extraction
# things like: \r\n mixed with \n, multiple spaces in a row, huge gaps of blank lines
# we need to clean these up so the text is consistent
# BUT we do NOT remove legal terms, punctuation, or anything meaningful
# the github doc says "preserve important legal terminology during cleaning"

def clean_contract_text(text):
    # some lines end with \r\n (Windows style) and some with just \n (Mac/Linux style)
    # make them all \n so the text is consistent
    text = text.replace('\r\n', '\n').replace('\r', '\n')

    # collapse runs of multiple spaces or tabs into a single space
    # e.g. "the    Company" becomes "the Company"
    # this happens a lot in contracts converted from PDF
    text = re.sub(r'[ \t]+', ' ', text)

    # collapse 3+ blank lines in a row into just 2
    # contracts often have huge gaps between sections from the PDF layout
    # we keep double line breaks (they mark section boundaries) but remove excessive ones
    text = re.sub(r'\n{3,}', '\n\n', text)

    # remove any leading/trailing whitespace from the whole document
    return text.strip()

# apply that cleaning to every contract's full text
contracts_df["context"] = contracts_df["context"].apply(clean_contract_text)

# print some stats so we can sanity check the contracts look reasonable
# min should be around 1,000 chars (a very short contract)
# max should be around 300,000 chars (a very long one)
print(f"Contract lengths (chars):")
print(f"  Min: {contracts_df['context'].str.len().min()}")
print(f"  Max: {contracts_df['context'].str.len().max()}")
print(f"  Avg: {contracts_df['context'].str.len().mean():.0f}")

In [ ]:
# Standardize contract IDs, clause labels, annotation spans, and document metadata

print("--- Contract IDs Standardization ---")
# Contract IDs (titles) are used as unique identifiers across the dataset.
# The 'contract_title' column serves this purpose, extracted directly from source JSON.
# They are used as provided by the original dataset after loading.
print(f"Number of unique contract titles in contracts_df: {contracts_df['contract_title'].nunique()}")
print(f"Sample contract titles: {contracts_df['contract_title'].head(2).tolist()}")

print("\n--- Clause Labels Standardization ---")
# Clause labels were standardized by creating a 'category' column from 'category_raw',
# removing numerical suffixes. This ensures that 'Parties_0', 'Parties_1', etc.,
# all map to 'Parties'.

def get_base_category(cat):
    # looks for a pattern like _0, _1, _2, _10 at the very end of the string
    # and removes it, leaving just the category name
    return re.sub(r'_\d+$', '', cat)

# Ensure the 'category' column exists for demonstration
if 'category' not in annotations_df.columns:
    annotations_df["category"] = annotations_df["category_raw"].apply(get_base_category)

print(f"Number of unique raw categories: {annotations_df['category_raw'].nunique()}")
print(f"Number of unique standardized categories: {annotations_df['category'].nunique()}")
print("Top 5 standardized categories and their counts:")
print(annotations_df['category'].value_counts().head())

print("\n--- Annotation Spans Standardization ---")
# Annotation spans (clause_text) were cleaned in a previous step (cell uvZTjGG_TqIV)
# by removing short/blank entries and stripping leading/trailing whitespace.
# 'answer_start' is a numerical index, inherently standardized.
print(f"Length of annotations_df after cleaning clause_text: {len(annotations_df)}")
print(f"Sample cleaned clause text: '{annotations_df['clause_text'].iloc[0]}'"
)
print(f"Type of answer_start: {annotations_df['answer_start'].dtype}")

print("\n--- Document Metadata (Contract Text) Standardization ---")
# Document metadata (full contract text in 'context' column of contracts_df)
# was standardized in a previous step (cell xj2-4AxJl31E) by:
# - Normalizing line endings (\r\n to \n)
# - Collapsing multiple spaces/tabs into single spaces
# - Collapsing excessive blank lines (3+ into 2)
# This improves consistency without altering legal terminology.
print(f"Sample of cleaned contract text (first 200 chars):")
print(f"'{contracts_df['context'].iloc[0][:200]}...'"
)
print(f"Average contract text length after cleaning: {contracts_df['context'].str.len().mean():.0f} characters")

In [ ]:
# Load contract titles for the official test set
with open("/content/cuad_data/test.json", "r") as f:
    test_data = json.load(f)
test_contract_titles = [item["title"] for item in test_data["data"]]

# Load contract titles for the official training set
with open("/content/cuad_data/train_separate_questions.json", "r") as f:
    train_data = json.load(f)
train_contract_titles = [item["title"] for item in train_data["data"]]

# Create the official test set annotations DataFrame
test_annotations_df = annotations_df[annotations_df["contract_title"].isin(test_contract_titles)].copy()

# Create the combined training and validation set annotations DataFrame
train_val_annotations_df = annotations_df[annotations_df["contract_title"].isin(train_contract_titles)].copy()

print(f"Official Test Set Annotations: {len(test_annotations_df)}")
print(f"Combined Train/Validation Set Annotations: {len(train_val_annotations_df)}")

# Get unique contract titles from the combined train/validation set for a contract-wise split
unique_train_val_contracts = train_val_annotations_df["contract_title"].unique()

# Split the unique contract titles into training and validation subsets
train_contract_subset, val_contract_subset = train_test_split(
    unique_train_val_contracts, test_size=0.2, random_state=42
)

# Filter the combined train/validation annotations DataFrame based on these subsets
train_annotations_df = train_val_annotations_df[train_val_annotations_df["contract_title"].isin(train_contract_subset)].copy()
val_annotations_df = train_val_annotations_df[train_val_annotations_df["contract_title"].isin(val_contract_subset)].copy()

print(f"\nFinal Training Set Annotations: {len(train_annotations_df)}")
print(f"Validation Set Annotations: {len(val_annotations_df)}")

# Display the number of unique contracts in each set
print(f"\nNumber of unique contracts in Official Test Set: {test_annotations_df['contract_title'].nunique()}")
print(f"Number of unique contracts in Final Training Set: {train_annotations_df['contract_title'].nunique()}")
print(f"Number of unique contracts in Validation Set: {val_annotations_df['contract_title'].nunique()}")

# Also create corresponding splits for the contracts_df
test_contracts_df = contracts_df[contracts_df["contract_title"].isin(test_contract_titles)].copy()
train_contracts_df = contracts_df[contracts_df["contract_title"].isin(train_contract_subset)].copy()
val_contracts_df = contracts_df[contracts_df["contract_title"].isin(val_contract_subset)].copy()

print(f"\nNumber of contracts in Official Test Set (contracts_df): {len(test_contracts_df)}")
print(f"Number of contracts in Final Training Set (contracts_df): {len(train_contracts_df)}")
print(f"Number of contracts in Validation Set (contracts_df): {len(val_contracts_df)}")

In [ ]:
# save the final files so teammates can load them directly
train_contracts_df.to_csv("train_contracts.csv", index=False)
val_contracts_df.to_csv("val_contracts.csv", index=False)
test_contracts_df.to_csv("test_contracts.csv", index=False)
train_annotations_df.to_csv("train_annotations.csv", index=False)
val_annotations_df.to_csv("val_annotations.csv", index=False)
test_annotations_df.to_csv("test_annotations.csv", index=False)

print("Saved 6 files:")
print(f"  train_contracts.csv  ({len(train_contracts_df)} contracts)")
print(f"  val_contracts.csv    ({len(val_contracts_df)} contracts)")
print(f"  test_contracts.csv   ({len(test_contracts_df)} contracts)")
print(f"  train_annotations.csv ({len(train_annotations_df)} annotations)")
print(f"  val_annotations.csv   ({len(val_annotations_df)} annotations)")
print(f"  test_annotations.csv  ({len(test_annotations_df)} annotations)")

In [ ]:
## Task 2: Run EDA  — Hadent & Vrinda

In [ ]:
df_dict = {
    "train_contracts_df": train_contracts_df,
    "val_contracts_df": val_contracts_df,
    "test_contracts_df": test_contracts_df,
    "train_annotations_df": train_annotations_df,
    "val_annotations_df": val_annotations_df,
    "test_annotations_df": test_annotations_df
}

In [ ]:
for df_name, df in df_dict.items():
    print(f"--- {df_name} ---")
    print("Shape:", df.shape)
    print(list(df.columns))

In [ ]:
#check split proportions
total= len(train_contracts_df)+len(val_contracts_df)+ len(test_contracts_df)
print("Train:", len(train_contracts_df)/total)
print("Val:", len(val_contracts_df)/total)
print("Test:", len(test_contracts_df)/total)

In [ ]:
#checking if data leaked
train_titles = set(train_contracts_df["contract_title"])
val_titles = set(val_contracts_df["contract_title"])
test_titles = set(test_contracts_df["contract_title"])

print("Train-Val overlap:", len(train_titles & val_titles))
print("Train-Test overlap:", len(train_titles & test_titles))
print("Val-Test overlap:", len(val_titles & test_titles))


In [ ]:
#compare contract lengths across train/val/test
for name, df in {
    "Train": train_contracts_df,
    "Validation": val_contracts_df,
    "Test": test_contracts_df
}.items():
    word_counts = df["context"].str.split().str.len()
    print(f"\n{name}")
    print("Average:", word_counts.mean())
    print("Median:", word_counts.median())
    print("Min:", word_counts.min())
    print("Max:", word_counts.max())

In [ ]:
#checking content-specific statistics
train_contracts_df["word_count"] = (train_contracts_df["context"].str.split().str.len())
val_contracts_df["word_count"] = (val_contracts_df["context"].str.split().str.len())
test_contracts_df["word_count"] = (test_contracts_df["context"].str.split().str.len())
contract_dfs = {
    "Train": train_contracts_df,
    "Validation": val_contracts_df,
    "Test": test_contracts_df}

for name, df in contract_dfs.items():
    print(f"--- {name} ---")
    print(df["word_count"].describe())
    print()

In [ ]:
#checking content-length distribution
import matplotlib.pyplot as plt

plt.hist(train_contracts_df["word_count"], bins=30)
plt.xlabel("Contract Length (Words)")
plt.ylabel("Number of Contracts")
plt.title("Distribution of Training Contract Lengths")
plt.show()

In [ ]:
#bar chart showing train/val/test split
splits = ["Train", "Validation", "Test"]
split_counts = [
    len(train_contracts_df),
    len(val_contracts_df),
    len(test_contracts_df)]

plt.bar(splits, split_counts)

plt.xlabel("Dataset Split")
plt.ylabel("Number of Contracts")
plt.title("Number of Contracts by Dataset Split")

plt.show()

In [ ]:
#view longest contracts

longest = (train_contracts_df.sort_values("word_count", ascending=False).head(10))

plt.barh(longest["contract_title"], longest["word_count"])
plt.xlabel("Word Count")
plt.ylabel("Contract")
plt.title("10 Longest Contracts in the Training Set")
plt.gca().invert_yaxis()
plt.show()

In [ ]:
category_distribution = pd.DataFrame({
    "train": train_annotations_df["category"].value_counts(),
    "val": val_annotations_df["category"].value_counts(),
    "test": test_annotations_df["category"].value_counts()
}).fillna(0)

print(category_distribution)

In [ ]:
category_proportions = pd.DataFrame({
    "train": train_annotations_df["category"].value_counts(normalize=True),
    "val": val_annotations_df["category"].value_counts(normalize=True),
    "test": test_annotations_df["category"].value_counts(normalize=True)
}).fillna(0)

print(category_proportions.round(3))

In [ ]:
train_annotations_df["category"].value_counts().plot(kind="bar", figsize=(12, 6), title="Category Distribution in Training Set")

In [ ]:
val_annotations_df["category"].value_counts().plot(kind="bar", figsize=(12, 6), title="Category Distribution in Validation Set")

In [ ]:
test_annotations_df["category"].value_counts().plot(kind="bar", figsize=(12, 6), title="Category Distribution in Test Set")

In [ ]:
# Task 3: Divide the contracts into chunks/sections  & categorise the chunks. (Abdoul, Aastha)

1) Chunking

In [ ]:
print(train_contracts_df.columns)

print("\nShape:")
print(train_contracts_df.shape)

print("\nFirst row:")
print(train_contracts_df.iloc[0])

In [ ]:
# Load the BERT tokenizer that will be used for chunking.
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print("Model maximum length:", tokenizer.model_max_length)

In [ ]:
# Chunking settings

CHUNK_SIZE = 450

# Keep 50 tokens from the previous chunk to preserve data that might fail between two chunks.

OVERLAP = 50


STEP_SIZE = CHUNK_SIZE - OVERLAP

print("Chunk size:", CHUNK_SIZE)
print("Overlap:", OVERLAP)
print("Step size:", STEP_SIZE)

In [ ]:
# Function to divide one contract into overlapping token-based chunks.
def chunk_contract(text):


    token_ids = tokenizer.encode(
        text,
        add_special_tokens=False,
        truncation=False
    )

    chunks = []


    for start in range(0, len(token_ids), STEP_SIZE):


        chunk_tokens = token_ids[start:start + CHUNK_SIZE]

        # Convert the token IDs back into readable text.
        chunk_text = tokenizer.decode(
            chunk_tokens,
            skip_special_tokens=True
        )

        final_tokens = tokenizer.encode(
            chunk_text,
            add_special_tokens=False,
            truncation=True,
            max_length=CHUNK_SIZE
        )


        chunk_text = tokenizer.decode(
            final_tokens,
            skip_special_tokens=True
        )


        chunks.append(chunk_text)

        # Stop once the final tokens have been included.
        if start + CHUNK_SIZE >= len(token_ids):
            break

    return chunks

In [ ]:
# Testing the chunking function on one training contract.

sample_contract = train_contracts_df.iloc[0]["context"]

sample_chunks = chunk_contract(sample_contract)

print("Original contract tokens:",
      len(tokenizer.encode(sample_contract, add_special_tokens=False)))

print("Number of chunks:", len(sample_chunks))

for i, chunk in enumerate(sample_chunks):
    token_count = len(
        tokenizer.encode(chunk, add_special_tokens=False)
    )

    print(f"Chunk {i + 1}: {token_count} tokens")

In [ ]:
# Verify that consecutive chunks have the 50-tokens overlap.

sample_token_ids = tokenizer.encode(
    sample_contract,
    add_special_tokens=False,
    truncation=False
)

chunk_1_tokens = sample_token_ids[0:450]
chunk_2_tokens = sample_token_ids[400:850]

overlap_matches = chunk_1_tokens[-OVERLAP:] == chunk_2_tokens[:OVERLAP]

print("Expected overlap:", OVERLAP)
print("Overlap is correct:", overlap_matches)

In [ ]:
# Apply the chunking function to every dataset contract.

def create_chunk_dataframe(contracts_df):

    chunk_rows = []

    for _, row in contracts_df.iterrows():

        contract_title = row["contract_title"]
        contract_text = row["context"]

        chunks = chunk_contract(contract_text)

        for chunk_number, chunk_text in enumerate(chunks):

            chunk_rows.append({
                "contract_title": contract_title,
                "chunk_id": chunk_number,
                "chunk_text": chunk_text
            })

    return pd.DataFrame(chunk_rows)

In [ ]:
# Create chunks for the training contracts.
train_chunks_df = create_chunk_dataframe(train_contracts_df)

print("Training contracts:", len(train_contracts_df))
print("Training chunks:", len(train_chunks_df))

print("\nColumns:")
print(train_chunks_df.columns)

print("\nFirst few chunks:")
display(train_chunks_df.head())

In [ ]:
# Create chunks for the validation and test contracts.

val_chunks_df = create_chunk_dataframe(val_contracts_df)
test_chunks_df = create_chunk_dataframe(test_contracts_df)

print("Training chunks:", len(train_chunks_df))
print("Validation chunks:", len(val_chunks_df))
print("Test chunks:", len(test_chunks_df))

print("\nTotal chunks:",
      len(train_chunks_df) +
      len(val_chunks_df) +
      len(test_chunks_df))

In [ ]:
# Verify that no contract appears in more than one dataset.

train_titles = set(train_chunks_df["contract_title"])
val_titles = set(val_chunks_df["contract_title"])
test_titles = set(test_chunks_df["contract_title"])

# Check for contracts shared between datasets.
train_val_overlap = train_titles.intersection(val_titles)
train_test_overlap = train_titles.intersection(test_titles)
val_test_overlap = val_titles.intersection(test_titles)

print("Train/Validation shared contracts:", len(train_val_overlap))
print("Train/Test shared contracts:", len(train_test_overlap))
print("Validation/Test shared contracts:", len(val_test_overlap))

In [ ]:
# Final check

def check_chunks(chunks_df, dataset_name):

    empty_chunks = chunks_df["chunk_text"].isna().sum()
    empty_chunks += (chunks_df["chunk_text"].str.strip() == "").sum()

    token_lengths = chunks_df["chunk_text"].apply(
        lambda text: len(
            tokenizer.encode(text, add_special_tokens=False)
        )
    )

    print(f"{dataset_name}:")
    print("Total chunks:", len(chunks_df))
    print("Empty chunks:", empty_chunks)
    print("Maximum chunk size:", token_lengths.max())
    print("Minimum chunk size:", token_lengths.min())
    print("Chunks over 450 tokens:", (token_lengths > CHUNK_SIZE).sum())
    print()


check_chunks(train_chunks_df, "Training")
check_chunks(val_chunks_df, "Validation")
check_chunks(test_chunks_df, "Test")

Categorise the chunks

In [ ]:
# Get the original contract texts from the dataset loaded earlier.
original_contexts = {
    contract["title"]: contract["paragraphs"][0]["context"]
    for contract in data["data"]
}


# Match each chunk with its clause categories.
def categorise_chunks(chunks_df, contracts_df, annotations_df):

    result = chunks_df.copy().reset_index(drop=True)
    categories = [[] for _ in range(len(result))]
    unmatched = []

    # Go through each contract separately.
    for _, contract in contracts_df.iterrows():

        title = contract["contract_title"]
        text = contract["context"]
        original_text = original_contexts[title]

        # Get each token's position in the contract.
        tokens = tokenizer(
            text,
            add_special_tokens=False,
            truncation=False,
            return_offsets_mapping=True
        )

        positions = tokens["offset_mapping"]

        # Get the annotations for this contract.
        annotations = annotations_df[
            annotations_df["contract_title"] == title
        ].drop_duplicates(
            subset=["category", "answer_start", "clause_text"]
        )

        clauses = []

        for _, annotation in annotations.iterrows():

            # Match the spacing used in the earlier cleaning step.
            clause = clean_contract_text(annotation["clause_text"])
            original_start = int(annotation["answer_start"])

            expected_start = len(
                clean_contract_text(original_text[:original_start])
            )

            # Find the annotated occurrence near its original location.
            # Cleaning can leave up to 3 spaces or line breaks here.
            start = text.find(
                clause,
                max(0, expected_start - 3),
                expected_start + len(clause) + 4
            )

            if not clause or start == -1:
                unmatched.append({
                    "contract_title": title,
                    "category": annotation["category"],
                    "clause_text": annotation["clause_text"],
                    "answer_start": original_start
                })
                continue

            clauses.append({
                "category": annotation["category"],
                "start": start,
                "end": start + len(clause)
            })

        # Get this contract's chunks.
        contract_chunks = result[
            result["contract_title"] == title
        ]

        for index, chunk in contract_chunks.iterrows():

            # Use the same token windows as the chunking function.
            first_token = int(chunk["chunk_id"]) * STEP_SIZE
            last_token = min(
                first_token + CHUNK_SIZE,
                len(positions)
            )

            chunk_start = positions[first_token][0]
            chunk_end = positions[last_token - 1][1]

            # Keep every category whose clause overlaps the chunk.
            categories[index] = sorted({
                clause["category"]
                for clause in clauses
                if clause["start"] < chunk_end
                and clause["end"] > chunk_start
            })

    result["categories"] = categories
    result["num_categories"] = result["categories"].apply(len)

    return result, pd.DataFrame(
        unmatched,
        columns=[
            "contract_title", "category",
            "clause_text", "answer_start"
        ]
    )

In [ ]:
# Categorise the training chunks.
train_chunks_df, train_unmatched = categorise_chunks(
    train_chunks_df,
    train_contracts_df,
    train_annotations_df
)

# Categorise the validation chunks.
val_chunks_df, val_unmatched = categorise_chunks(
    val_chunks_df,
    val_contracts_df,
    val_annotations_df
)

# Categorise the test chunks.
test_chunks_df, test_unmatched = categorise_chunks(
    test_chunks_df,
    test_contracts_df,
    test_annotations_df
)

In [ ]:
# Check how many chunks received categories.
for name, chunks, unmatched in [
    ("Training", train_chunks_df, train_unmatched),
    ("Validation", val_chunks_df, val_unmatched),
    ("Test", test_chunks_df, test_unmatched)
]:
    print(f"\n{name}:")
    print("Total chunks:", len(chunks))
    print("Chunks with categories:",
          (chunks["num_categories"] > 0).sum())
    print("Chunks without annotations:",
          (chunks["num_categories"] == 0).sum())
    print("Unmatched annotations:", len(unmatched))

    if not unmatched.empty:
        display(unmatched.head())

# Stop here if any annotations need checking.
if any(not frame.empty for frame in [
    train_unmatched, val_unmatched, test_unmatched
]):
    raise ValueError("Review the unmatched annotations before saving.")

# Show a few categorised chunks.
display(train_chunks_df[
    ["contract_title", "chunk_id", "chunk_text",
     "categories", "num_categories"]
].head())

In [ ]:
# Task 4: TF/IDF keyword baseline model (Kenneth & John)

# Building a TF-IDF Model. TF is Text-Frequency to show how important
# specific words are in chunks based on their frequency.
# IDF is Inverse Document Frequency to check how informative one word is
# across ALL chunks.

from sklearn.feature_extraction.text import TfidfVectorizer # Text to Numbers
from sklearn.preprocessing import MultiLabelBinarizer # Category lists into unique binary numbers
from sklearn.multiclass import OneVsRestClassifier # handles the 41 independent category predictions
from sklearn.linear_model import LogisticRegression # Logistic Regression Classifier
from sklearn.metrics import classification_report # Accuracy metrics (precision, recall, F1)

mlb = MultiLabelBinarizer()

# transform training, valuation, and test data into multi label binary
y_train = mlb.fit_transform(train_chunks_df["categories"]) # learn label structure through fit_transform
y_val = mlb.transform(val_chunks_df["categories"])
y_test = mlb.transform(test_chunks_df["categories"])

print(y_train.shape)
print(mlb.classes_)

In [ ]:
tfidf = TfidfVectorizer(
    lowercase=True, # standardize capitalization
    ngram_range=(1,2), # create features using both one and two word sequences
    min_df=2, # ignore terms that occur in fewer than 2 chunks
    max_features=50000 # keep up to 50,000 most relevant terms/phrases
)

# learns the vocabulary and IDF converts each chunk into numerical values
X_train = tfidf.fit_transform(train_chunks_df["chunk_text"])
X_val = tfidf.transform(val_chunks_df["chunk_text"])
X_test = tfidf.transform(test_chunks_df["chunk_text"])

feature_names = tfidf.get_feature_names_out()
# prints out the first 20 words learned
print(feature_names[:20])

chunk_index = 5
row = X_train[chunk_index] # selects the numerical TF-IDF representation for chunk 5

scores = row.toarray()[0] # converts single row into a normal array to inspect


top_indices = scores.argsort()[-15:][::-1] # positions of the 15 largest TF-IDF scores


for index in top_indices:
    print(feature_names[index], scores[index])




In [ ]:
# What's left:
# 1) Train a baseline classifier using Logistic regression
#     Since this is a multi-label problem, use OneVsRestClassifier around Logistic Regression
#     Consider using class_weights = 'balanced' because the categories are a bit unbalanced
#     Train on X_train/y_train and predic on X_val
# 2) Evaluate per-category precision, recall, and F1 with classification_report

In [ ]:
X_train, y_train.shape

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report

# 1) Build the model
base_lr = LogisticRegression(
    class_weight='balanced',   # handles class imbalance per label
    solver='liblinear',        # good for large sparse data; 'saga' is another option
    max_iter=1000,
    C=1.0
)
clf = OneVsRestClassifier(base_lr, n_jobs=-1)  # n_jobs=-1 fits the 41 labels in parallel

# 2) Train
clf.fit(X_train, y_train)

# 3) Predict on validation
y_val_pred = clf.predict(X_val)               # binary label matrix (n_val, 41)
y_val_proba = clf.predict_proba(X_val)        # probabilities, useful for threshold tuning

# 4) Evaluate
print("Micro F1:   ", f1_score(y_val, y_val_pred, average='micro'))
print("Macro F1:   ", f1_score(y_val, y_val_pred, average='macro'))
print("Samples F1: ", f1_score(y_val, y_val_pred, average='samples'))
print("Micro precision:", precision_score(y_val, y_val_pred, average='micro'))
print("Micro recall:   ", recall_score(y_val, y_val_pred, average='micro'))

# Per-label breakdown (pass target_names=label_names if you have them)
print(classification_report(y_val, y_val_pred, zero_division=0))

In [ ]:
#Milestone 2: Task 1 (Vrinda and John)

In [ ]:
!pip install -q transformers datasets accelerate

import torch
import numpy as np

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from datasets import Dataset
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    classification_report
)

In [ ]:
#load bert-base-uncased model
model_name = "bert-base-uncased"

bert_tokenizer = AutoTokenizer.from_pretrained(model_name)

num_labels = len(mlb.classes_)

bert_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    problem_type="multi_label_classification",
    id2label={
        i: label for i, label in enumerate(mlb.classes_)
    },
    label2id={
        label: i for i, label in enumerate(mlb.classes_)
    }
)

print("Number of labels:", num_labels)
print("GPU available:", torch.cuda.is_available())

In [ ]:
#prepare the data for bert and print training, test, and validation chunks
def prepare_bert_dataset(chunks_df, labels):

    dataset = Dataset.from_dict({
        "text": chunks_df["chunk_text"].tolist(),
        "labels": labels.astype(np.float32).tolist()
    })

    def tokenize(batch):
        return bert_tokenizer(
            batch["text"],
            truncation=True,
            padding="max_length",
            max_length=512
        )

    dataset = dataset.map(
        tokenize,
        batched=True
    )

    dataset = dataset.remove_columns(["text"])
    return dataset

bert_train = prepare_bert_dataset(
    train_chunks_df, y_train
)

bert_val = prepare_bert_dataset(
    val_chunks_df, y_val
)

bert_test = prepare_bert_dataset(
    test_chunks_df, y_test
)

print("Training chunks:", len(bert_train))
print("Validation chunks:", len(bert_val))
print("Test chunks:", len(bert_test))

In [ ]:
#create the validation metrics
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    probabilities = torch.sigmoid(
        torch.tensor(logits)
    ).numpy()

    predictions = (probabilities >= 0.5).astype(int)

    return {
        "micro_f1": f1_score(
            labels, predictions,
            average="micro",
            zero_division=0
        ),
        "macro_f1": f1_score(
            labels, predictions,
            average="macro",
            zero_division=0
        ),
        "precision": precision_score(
            labels, predictions,
            average="micro",
            zero_division=0
        ),
        "recall": recall_score(
            labels, predictions,
            average="micro",
            zero_division=0
        )
    }

In [ ]:
#bert training
training_args = TrainingArguments(

    output_dir="/content/bert_clause_model",

    #evaluate and save after each epoch
    eval_strategy="epoch",
    save_strategy="epoch",

    #learning rate for BERT
    learning_rate=2e-5,

    #num of chunks per batch
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    #batch size
    gradient_accumulation_steps=2,

    #1 epoch
    num_train_epochs=1,

    weight_decay=0.01,

    logging_steps=100,


    load_best_model_at_end=True,
    metric_for_best_model="micro_f1",
    greater_is_better=True,

    save_total_limit=2,

    fp16=torch.cuda.is_available(),

    report_to="none"
)


#create the Trainer
trainer = Trainer(
    model=bert_model,
    args=training_args,
    train_dataset=bert_train,
    eval_dataset=bert_val,
    compute_metrics=compute_metrics
)

#begin training
trainer.train()

In [ ]:
#generate predictions on the validation dataset
val_results = trainer.predict(bert_val)

#convert model outputs into probabilities
val_probabilities = torch.sigmoid(
    torch.tensor(val_results.predictions)
).numpy()

#convert probabilities into predicted labels
val_predictions = (val_probabilities >= 0.5).astype(int)

#display evaluation metrics
print("BERT VALIDATION RESULTS")

print("Micro-F1:", f1_score(
    y_val, val_predictions,
    average="micro",
    zero_division=0
))

print("Macro-F1:", f1_score(
    y_val, val_predictions,
    average="macro",
    zero_division=0
))

print("Precision:", precision_score(
    y_val, val_predictions,
    average="micro",
    zero_division=0
))

print("Recall:", recall_score(
    y_val, val_predictions,
    average="micro",
    zero_division=0
))

print(classification_report(
    y_val,
    val_predictions,
    target_names=mlb.classes_,
    zero_division=0
))

In [ ]:
#display the Trainer's training history
print(trainer.state.log_history)

#check the current training step
print("Global training step:", trainer.state.global_step)

In [ ]:
# Milestone 2: Task 2 Kenneth (Addressing Class Imbalances)

In [ ]:
# Analyze and Visualize Current Class Distributions
import seaborn as sns
# Retrieve categories and compute frequency counts at chunk level
categories = mlb.classes_
train_counts = y_train.sum(axis=0)
val_counts = y_val.sum(axis=0)

# 2. Compute percentages
train_pct = (train_counts / y_train.shape[0]) * 100
val_pct = (val_counts / y_val.shape[0]) * 100

# 3. Combine into a comparison DataFrame, sort by train count
dist_df = pd.DataFrame({
    'Category': categories,
    'Train_Count': train_counts,
    'Train_Pct': train_pct,
    'Val_Count': val_counts,
    'Val_Pct': val_pct
}).sort_values(by='Train_Count', ascending=False).reset_index(drop=True)

print("--- Chunk-Level Class Distribution Comparison ---")
display(dist_df.head(10)) # high count extreme
display(dist_df.tail(10)) # low count extreme

# 4. Highlight extreme imbalances
majority_class = dist_df.iloc[0]
minority_class = dist_df.iloc[-1]
print(f"\nMajority Category: '{majority_class['Category']}' with {majority_class['Train_Count']} occurrences ({majority_class['Train_Pct']:.2f}% of chunks)")
print(f"Minority Category: '{minority_class['Category']}' with {minority_class['Train_Count']} occurrences ({minority_class['Train_Pct']:.2f}% of chunks)")
print(f"Imbalance Ratio (Majority / Minority): {majority_class['Train_Count'] / (minority_class['Train_Count'] + 1e-5):.1f}x")

# 5. Comparative Bar Chart
plt.figure(figsize=(14, 8))
plot_data = pd.melt(dist_df, id_vars=['Category'], value_vars=['Train_Count', 'Val_Count'],
                    var_name='Dataset', value_name='Count')

sns.barplot(x='Count', y='Category', hue='Dataset', data=plot_data, palette='viridis')
plt.title('Comparative Chunk-Level Category Frequencies (Train vs Validation)')
plt.xlabel('Number of Chunks')
plt.ylabel('Clause Category')
plt.tight_layout()
plt.show()

In [ ]:
# Step 1: Calculate Positive Class Weights (pos_weight) for BCEWithLogitsLoss

total_samples = y_train.shape[0]
pos_counts = y_train.sum(axis=0)
neg_counts = total_samples - pos_counts

# Standard positive weights formula
# We add a small epsilon to avoid division by zero if a category is completely empty
pos_weights = neg_counts / (pos_counts + 1e-8)

# Convert to a PyTorch tensor on the appropriate device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pos_weights_tensor = torch.tensor(pos_weights, dtype=torch.float32).to(device)

# 2. Print Diagnostic Information
weight_df = pd.DataFrame({
    'Category': categories,
    'Count': pos_counts,
    'Weight': pos_weights
}).sort_values(by='Weight', ascending=False).reset_index(drop=True)

print("--- Top 10 Highest-Weighted Categories (Rare Categories) ---")
display(weight_df.head(10))

print("\n--- Top 10 Lowest-Weighted Categories (Frequent Categories) ---")
display(weight_df.tail(10))

# Ensure the tensor length matches our 41 classes
assert len(pos_weights_tensor) == len(categories), "Weights dimensions do not match the number of categories!"
print(f"\nSuccessfully constructed weight tensor on device '{device}' with shape {pos_weights_tensor.shape}")

In [ ]:
# Step 2: Implement and Validate a Custom Loss Trainer

import torch.nn as nn

class WeightedMultilabelTrainer(Trainer):
    """
    Custom Hugging Face Trainer subclass that handles multi-label classification
    with severe class imbalances using positive class weights (pos_weight) in
    BCEWithLogitsLoss.

    Attributes:
        pos_weights (torch.Tensor): A tensor containing calculated positive class weights
                                    for each category to scale positive class losses.
        loss_fn (nn.Module): The custom loss function using BCEWithLogitsLoss.
    """
    def __init__(self, pos_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        # Save class weights to local attribute
        self.pos_weights = pos_weights

        # Initialize BCEWithLogitsLoss with positive class weights
        # This ensures rare positive classes are penalized more when missed,
        # balancing out the majority of negative instances.
        self.loss_fn = nn.BCEWithLogitsLoss(pos_weight=self.pos_weights)

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        """
        Calculates custom loss using Weighted BCEWithLogitsLoss instead of the default loss function.

        Args:
            model (PreTrainedModel): The PyTorch multi-label classification model.
            inputs (dict): Dictionary containing input IDs, attention masks, and target labels.
            return_outputs (bool, optional): If True, returns both loss and outputs.
            num_items_in_batch (int, optional): Number of items in batch.

        Returns:
            loss (torch.Tensor) or (loss, outputs):
        """
        # Extract target labels from the batch dictionary
        labels = inputs.get("labels")

        # Standard forward pass to obtain model logits
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # Calculate custom weighted loss using BCEWithLogitsLoss
        loss = self.loss_fn(logits, labels)

        return (loss, outputs) if return_outputs else loss

# Sanity Check: Test the weighted trainer initialization
print("WeightedMultilabelTrainer class successfully defined. Verifying initialization...")

temp_training_args = TrainingArguments(
    output_dir="/content/temp_test_dir",
    report_to="none",
    disable_tqdm=True
)

test_trainer = WeightedMultilabelTrainer(
    pos_weights=pos_weights_tensor,
    model=bert_model,
    args=temp_training_args,
    train_dataset=bert_train.select(range(10)),
    eval_dataset=bert_val.select(range(10))
)

print("Sanity Check: WeightedMultilabelTrainer successfully initialized!")

In [ ]:
Task #4

In [ ]:
EXPORT_INPUTS = True

if EXPORT_INPUTS:
    import json
    import numpy as np
    from datetime import datetime, timezone
    from google.colab import files

    MODEL_SELECTION = "both"
    RUN_DESCRIPTION = ""
    ALIGNMENT_CONFIRMED = False

    if not ALIGNMENT_CONFIRMED or not RUN_DESCRIPTION.strip():
        raise ValueError("Confirm alignment and fill RUN_DESCRIPTION before export.")
    choices = {"tfidf": ("y_val_pred", "y_val_proba"),
               "bert": ("val_predictions", "val_probabilities")}
    if MODEL_SELECTION not in ("tfidf", "bert", "both"):
        raise ValueError("Choose tfidf, bert, or both.")
    selected = list(choices) if MODEL_SELECTION == "both" else [MODEL_SELECTION]
    required = ["val_chunks_df", "y_val", "mlb"] + [choices[m][0] for m in selected]
    missing = [name for name in required if name not in globals()]
    if missing:
        raise ValueError("Missing runtime variables: " + ", ".join(missing) +
                         ". Run upstream cells or choose the available model.")

    chunks = val_chunks_df.reset_index(drop=True)
    truth = np.asarray(y_val)
    labels = [str(x) for x in mlb.classes_]
    if truth.shape != (len(chunks), len(labels)) or not np.isin(truth, [0, 1]).all():
        raise ValueError("Invalid label matrix or row/column alignment.")
    if not {"contract_title", "chunk_id", "chunk_text", "categories"}.issubset(chunks.columns):
        raise ValueError("Expected chunk metadata and categories are missing.")
    if not np.array_equal(mlb.transform(chunks["categories"]), truth):
        raise ValueError("Chunk annotations do not match y_val in this order.")
    records = []
    for _, row in chunks.iterrows():
        records.append({"contract_title": str(row["contract_title"]),
                        "chunk_id": str(row["chunk_id"]),
                        "chunk_text": str(row["chunk_text"])})
    models = {}
    for name in selected:
        pred_name, proba_name = choices[name]
        pred = np.asarray(globals()[pred_name])
        if pred.shape != truth.shape or not np.isin(pred, [0, 1]).all():
            raise ValueError(f"Invalid {name} predictions.")
        entry = {"predictions": pred.astype(int).tolist()}
        if proba_name in globals():
            proba = np.asarray(globals()[proba_name], dtype=float)
            if proba.shape != truth.shape or not np.isfinite(proba).all() or ((proba < 0) | (proba > 1)).any():
                raise ValueError(f"Invalid {name} probabilities.")
            entry["probabilities"] = proba.tolist()
        models[name] = entry
    bundle = {"schema_version": 1, "synthetic": False,
              "run_description": RUN_DESCRIPTION,
              "exported_at": datetime.now(timezone.utc).isoformat(),
              "labels": labels, "chunks": records,
              "y_true": truth.astype(int).tolist(), "models": models}
    with open("task4_inputs.json", "w") as f:
        json.dump(bundle, f, allow_nan=False)
    files.download("task4_inputs.json")
    print("Exported:", selected, "| chunks:", len(chunks), "| labels:", len(labels))

Next steps: initialize and train our custom WeightedMultilabelTrainer using the computed class weights tensor (pos_weights_tensor) and evaluate metrics now that classes are balanced.